In [1]:
!pip install -q vllm
!pip uninstall -y torchaudio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.2.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.4 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.2.2 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cu

In [2]:
!git clone https://github.com/VatsalRoy/keep-it-warm.git
%cd /kaggle/working/keep-it-warm
!python -m env.generate_world --seed 42

Cloning into 'keep-it-warm'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 119 (delta 40), reused 96 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 102.90 KiB | 7.92 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/kaggle/working/keep-it-warm
seed=42  world_sha256=4a5b9a1a8c73a5e39f83a7c00eb5d4ccafec0d2bf6912ce82e81f31f2f0dc419


In [3]:
import os, subprocess

cmd = [
    "vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507",
    "--dtype", "half",
    "--max-model-len", "28672",
    "--enable-prefix-caching",
    "--enable-prompt-tokens-details",
    "--gpu-memory-utilization", "0.90",
    "--port", "8000",
]
env = {**os.environ, "VLLM_SERVER_DEV_MODE": "1"}   # unlocks /reset_prefix_cache
log = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
print("started, pid", server.pid)

started, pid 142


In [4]:
import time, requests
for i in range(90):
    if server.poll() is not None:
        print("Server CRASHED - see log below"); break
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print(f"READY after ~{i*10}s"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("Not ready after 15 min - see log below")
!tail -n 15 vllm.log

READY after ~230s
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /generative_scoring, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /v1/messages/render, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /v1/completions/render, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /v1/responses/render, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /v1/chat/completions/derender, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:195] Route: /v1/completions/derender, Methods: POST
(APIServer pid=142) INFO 11-02 02:38:00 [launcher.py:1

In [5]:
print("reset_prefix_cache ->", requests.post("http://localhost:8000/reset_prefix_cache").status_code)
!grep -iE "KV cache|maximum concurrency" vllm.log

reset_prefix_cache -> 200
(EngineCore pid=218) INFO 11-02 02:36:23 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=218) INFO 11-02 02:37:21 [gpu_worker.py:759] Available KV cache memory: 4.08 GiB
(EngineCore pid=218) INFO 11-02 02:37:21 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8713 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9287. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=218) INFO 11-02 02:37:21 [kv_cache_utils.py:2327] GPU KV cache size: 29,728 tokens, Maximum concurrency for 28,672 tokens per request: 1.04x
(EngineCore pid=218) INFO 11-02 02:37:54 [gpu_worker.py:1034] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 7.87 GiB for consumed memory (weights

In [6]:
!python scripts/run_episodes.py --policy full --out runs/baseline_sessions

s01            s0 4/4 ✓✓✓✓  turns=20  end=completed        peak_ctx=6961   cached=94% $B=0.1079 wall=102.4s
s02            s0 2/4 ✓✓✗✗  turns=23  end=completed        peak_ctx=12230  cached=92% $B=0.1595 wall=180.6s
s03            s0 3/4 ✓✗✓✓  turns=37  end=completed        peak_ctx=20995  cached=95% $B=0.3437 wall=387.6s
s04            s0 0/4 ✗✗✗✗  turns=17  end=completed        peak_ctx=12400  cached=89% $B=0.1340 wall=135.3s
s05            s0 2/4 ✗✓✓✗  turns=27  end=completed        peak_ctx=7290   cached=94% $B=0.1174 wall=110.9s
s06            s0 1/4 ✗✗✗✓  turns=12  end=completed        peak_ctx=7150   cached=88% $B=0.0843 wall=89.5s
s07            s0 1/4 ✓✗✗✗  turns=22  end=completed        peak_ctx=5372   cached=95% $B=0.1089 wall=128.1s
s08            s0 2/4 ✗✓✓✗  turns=20  end=completed        peak_ctx=13129  cached=91% $B=0.1540 wall=151.9s
s09            s0 2/4 ✓✗✓✗  turns=30  end=completed        peak_ctx=11838  cached=95% $B=0.1882 wall=170.9s
s10            s0 0/4 ✗✗✗✗  t

In [7]:
!python scripts/summarize_run.py runs/baseline_sessions --episode s10

session        correct turns end              peak_ctx  cached       $A       $B  naive$B  wall_s
s01              4/4      20 completed            6961     94%   0.0080   0.1079   0.4803   102.4
s02              2/4      23 completed           12230     92%   0.0114   0.1595   0.6793   180.6
s03              3/4      37 completed           20995     95%   0.0313   0.3437   1.9531   387.6
s04              0/4      17 completed           12400     89%   0.0095   0.1340   0.5565   135.3
s05              2/4      27 completed            7290     94%   0.0086   0.1174   0.5177   110.9
s06              1/4      12 completed            7150     88%   0.0051   0.0843   0.2908    89.5
s07              1/4      22 completed            5372     95%   0.0070   0.1089   0.4150   128.1
s08              2/4      20 completed           13129     91%   0.0117   0.1540   0.7009   151.9
s09              2/4      30 completed           11838     95%   0.0161   0.1882   0.9906   170.9
s10              0/4